In [ ]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
%matplotlib inline

In [ ]:
# Define the directory you want to list files from
directory = '/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/FID20205_ECG/'

# Initialize an empty list to store file paths
file_paths = []
patient_id = []
visit_no = []

# Use os.walk to get all file paths
for root, dirs, files in os.walk(directory):
    for file in files:
        ids1 = file.split('.')[0]
        ids = ids1.split('_')[0]
        visit = ids1.split('_')[2]
        patient_id.append(int(ids))
        file_paths.append(os.path.join(root, file))
        visit_no.append(int(visit))
        

# Create a DataFrame from the list of file paths
df_ecg = pd.DataFrame(file_paths, columns=['FilePath_ECG'])
df_ecg['IID'] = patient_id
df_ecg['visit'] = visit_no
df_ecg = df_ecg.loc[df_ecg['visit'] == 2]
df_ecg = df_ecg.drop(['visit'], axis = 1)

cad_cases = pd.read_csv('cad_case_control.csv')
cad_cases = cad_cases.rename(columns={'Patient EID':'IID'})

import pandas as pd

# Load and prepare data
df_ecg = pd.merge(df_ecg, cad_cases, on='IID', how='inner')
df_test = pd.read_csv('Final_Datasets/test_data_incidence.csv')
controls_additional = pd.read_csv('Final_Datasets/imbalanced_control_iid.csv')
controls_additional = controls_additional[['IID', 'CAD', 'FilePath_ECG']]

# Remove test cases and controls
test_iids = set(df_test['IID'])

# Get all cases directly from df_ecg, excluding test data
cases = df_ecg[(df_ecg['CAD'] == 1) & (~df_ecg['IID'].isin(test_iids))]

# Get all controls specifically from controls_additional, excluding test data
controls = controls_additional[~controls_additional['IID'].isin(test_iids)]

# Combine cases and controls into an imbalanced dataset
df_imbalanced_train = pd.concat([cases, controls], axis=0).reset_index(drop=True)

# Save the imbalanced dataset
df_imbalanced_train.to_csv('Final_Datasets/train_imbalanced_ecg_temp.csv', sep=',', index=False)

# Report dataset statistics
cases_count = len(cases)
controls_count = len(controls)
print(f"Number of cases: {cases_count}")
print(f"Number of controls: {controls_count}")
print(f"Imbalance ratio (controls to cases): {controls_count / cases_count:.2f}")

In [ ]:
import pandas as pd

# Load and prepare data
df_ecg = pd.merge(df_ecg, cad_cases, on='IID', how='inner')
df_test = pd.read_csv('Final_Datasets/test_data_incidence.csv')
controls_additional = pd.read_csv('Final_Datasets/imbalanced_control_iid.csv')
controls_additional = controls_additional[['IID', 'CAD', 'FilePath_ECG']]

# Remove test cases and controls
test_iids = set(df_test['IID'])

# Get all cases directly from df_ecg, excluding test data
cases = df_ecg[(df_ecg['CAD'] == 1) & (~df_ecg['IID'].isin(test_iids))]

# Get all controls specifically from controls_additional, excluding test data
controls = controls_additional[~controls_additional['IID'].isin(test_iids)]

# Combine cases and controls into an imbalanced dataset
df_imbalanced_train = pd.concat([cases, controls], axis=0).reset_index(drop=True)

# Save the imbalanced dataset
df_imbalanced_train.to_csv('Final_Datasets/train_imbalanced_ecg_temp.csv', sep=',', index=False)

# Report dataset statistics
cases_count = len(cases)
controls_count = len(controls)
print(f"Number of cases: {cases_count}")
print(f"Number of controls: {controls_count}")
print(f"Imbalance ratio (controls to cases): {controls_count / cases_count:.2f}")

In [ ]:
df_ecg.head()

In [ ]:
df_imbalanced_train["CAD"].value_counts()

In [ ]:
df_imbalanced_train.head()

In [ ]:
df_test.head()

In [ ]:
# Function to extract ECG features from a given XML file
def extract_ecg_features(file_path):
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Extract Intervals and Durations
        pq_interval = root.findtext(".//PQInterval")
        p_duration = root.findtext(".//PDuration")
        qrs_duration = root.findtext(".//QRSDuration")
        qt_interval = root.findtext(".//QTInterval")
        qtc_interval = root.findtext(".//QTCInterval")
        rr_interval = root.findtext(".//RRInterval")
        pp_interval = root.findtext(".//PPInterval")

        # Convert to numeric values
        pq_interval = float(pq_interval) if pq_interval else None
        p_duration = float(p_duration) if p_duration else None
        qrs_duration = float(qrs_duration) if qrs_duration else None
        qt_interval = float(qt_interval) if qt_interval else None
        qtc_interval = float(qtc_interval) if qtc_interval else None
        rr_interval = float(rr_interval) if rr_interval else None
        pp_interval = float(pp_interval) if pp_interval else None

        # Extract Amplitudes from the MeasurementTable
        p_amplitudes = []
        q_amplitudes = []
        r_amplitudes = []
        t_amplitudes = []

        # Extract amplitude values from the 'MeasurementTable' for each lead
        measurement_table = root.find(".//MeasurementTable")
        if measurement_table is not None:
            p_wave = measurement_table.findtext("PAmplitude")
            q_wave = measurement_table.findtext("QAmplitude")
            r_wave = measurement_table.findtext("RAmplitude")
            t_wave = measurement_table.findtext("TAmplitude")

            # Convert to lists of floats
            p_amplitudes = [float(value) for value in p_wave.split(',') if value.strip()]
            q_amplitudes = [float(value) for value in q_wave.split(',') if value.strip()]
            r_amplitudes = [float(value) for value in r_wave.split(',') if value.strip()]
            t_amplitudes = [float(value) for value in t_wave.split(',') if value.strip()]

            # Calculate mean amplitudes
            mean_p_amplitude = np.mean(p_amplitudes) if p_amplitudes else None
            mean_q_amplitude = np.mean(q_amplitudes) if q_amplitudes else None
            mean_r_amplitude = np.mean(r_amplitudes) if r_amplitudes else None
            mean_t_amplitude = np.mean(t_amplitudes) if t_amplitudes else None
        else:
            mean_p_amplitude = mean_q_amplitude = mean_r_amplitude = mean_t_amplitude = None

        # Return extracted features
        return {
            'PQInterval': pq_interval,
            'PDuration': p_duration,
            'QRSDuration': qrs_duration,
            'QTInterval': qt_interval,
            'QTCInterval': qtc_interval,
            'RRInterval': rr_interval,
            'PPInterval': pp_interval,
            'MeanPAmplitude': mean_p_amplitude,
            'MeanQAmplitude': mean_q_amplitude,
            'MeanRAmplitude': mean_r_amplitude,
            'MeanTAmplitude': mean_t_amplitude
        }

    except ET.ParseError:
        # Handle parsing errors if any XML files are not well-formed
        print(f"Error parsing XML file: {file_path}")
        return {
            'PQInterval': None,
            'PDuration': None,
            'QRSDuration': None,
            'QTInterval': None,
            'QTCInterval': None,
            'RRInterval': None,
            'PPInterval': None,
            'MeanPAmplitude': None,
            'MeanQAmplitude': None,
            'MeanRAmplitude': None,
            'MeanTAmplitude': None
        }


# Initialize new columns in the DataFrame
for feature in ['PQInterval', 'PDuration', 'QRSDuration', 'QTInterval', 'QTCInterval', 'RRInterval', 'PPInterval',
                'MeanPAmplitude', 'MeanQAmplitude', 'MeanRAmplitude', 'MeanTAmplitude']:
    df_imbalanced_train[feature] = None

# Iterate over DataFrame rows and update with extracted features
for index, row in df_imbalanced_train.iterrows():
    features = extract_ecg_features(row['FilePath_ECG'])
    for feature_name, feature_value in features.items():
        df_imbalanced_train.at[index, feature_name] = feature_value

In [ ]:
df_imbalanced_train.head()

In [ ]:
df_imbalanced_train.isnull().sum()

In [ ]:
df_imbalanced_train = df_imbalanced_train.drop(['PQInterval', 'PDuration', 'MeanPAmplitude'], axis = 1)
df_imbalanced_train.isnull().sum()

In [ ]:
df_imbalanced_train = df_imbalanced_train.dropna()

In [ ]:
df_imbalanced_train.isnull().sum()

In [ ]:
df_imbalanced_train.to_csv('Final_Datasets/train_imbalanced_ecg.csv', index = False)

In [ ]:
df = pd.read_csv('Final_Datasets/train_imbalanced_ecg.csv')

In [ ]:
df.describe(include=[np.number])